In [1]:
import pickle
from collections import defaultdict
import os
from tqdm import tqdm
import re
import pandas as pd

In [3]:
test_file = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/QBioLiP/sequence_30_split/RNAL_test.pkl"
with open(test_file, "rb") as f:
    test_data = pickle.load(f)
print("Loaded test data:", len(test_data))

Loaded test data: 973


In [5]:
# ligs = defaultdict(int)
# rec_dir = "/n/netscratch/mzitnik_lab/Lab/afang/data/qbiolip/PL/redund_rec/"
# lig_dir = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/QBioLiP/PL/redund_lig_new/redund_lig/"
# out_dir = "/n/netscratch/mzitnik_lab/Lab/afang/data/qbiolip/PL/test_set_pdbs"
# for item in tqdm(test_data, total = len(test_data)):
#     lig_name = item['id'].split(".pdb_")[1]
#     rec_name = item['id'].split(".pdb_")[0] + ".pdb"
#     if not os.path.exists(os.path.join(lig_dir, lig_name)):
#         print("missing ligand", lig_name)
#     if not os.path.exists(os.path.join(rec_dir, rec_name)):
#         print("missing receptor", rec_name)
    
#     with open(os.path.join(rec_dir, rec_name), 'r') as f:
#         rec_lines = f.readlines()[:-1]
#     with open(os.path.join(lig_dir, lig_name), 'r') as f:
#         lig_lines = f.readlines()
    
#     with open(os.path.join(out_dir, item['id']), 'w') as f:
#         f.writelines(rec_lines)
#         f.writelines(lig_lines)

In [2]:
def parse_plip_output(file_name):
    # Read file content
    with open(file_name, 'r') as f:
        file_content = f.read()

    # Define regex patterns to identify sections and table rows
    section_pattern = r'\*\*(.+?)\*\*'
    row_pattern = r'\|\s*([\d]+)\s*\|\s*([A-Z]+)\s*\|\s*([A-Z])\s*\|\s*([\d]+)\s*\|\s*([A-Z0-9]+)\s*\|\s*([A-Z])\s*\|'

    # Dictionary to hold data for each bond type
    bond_data = []

    # Find all section headers
    sections = re.findall(section_pattern, file_content)

    # Iterate through each section
    for section in sections:
        section_start = file_content.find(f"**{section}**")
        section_end = file_content.find('**', section_start + len(section) + 4)
        
        # Extract the section contents
        if section_end == -1:
            section_end = len(file_content)
        
        section_content = file_content[section_start:section_end]

        # Find all matches in this section for the row pattern
        rows = re.findall(row_pattern, section_content)

        # Add bond type to each row and structure data
        bond_rows = [
            (
                int(resnr), restype, reschain, 
                int(resnr_lig), restype_lig, reschain_lig,
                section.strip()
            )
            for resnr, restype, reschain, resnr_lig, restype_lig, reschain_lig in rows
        ]

        bond_data.extend(bond_rows)

    # Create DataFrame
    df = pd.DataFrame(bond_data, columns=[
        'resnr', 'restype', 'reschain',
        'resnr_lig', 'restype_lig', 'reschain_lig',
        'bondtype'
    ])

    return df

In [6]:
plip_results = []
plip_dir = "/n/netscratch/mzitnik_lab/Lab/afang/data/qbiolip/PL/test_plip/"
for fname in tqdm(os.listdir(plip_dir), total=len(os.listdir(plip_dir))):
    plip_fname = os.path.join(plip_dir, fname, "report.txt")
    if not os.path.exists(plip_fname):
        print("missing", plip_fname)
        continue
    results = parse_plip_output(plip_fname)
    results['item_id'] = fname
    plip_results.append(results)
plip_results = pd.concat(plip_results)

plip_results.to_csv("PL_plip_results.csv", index=False)

100%|██████████| 6654/6654 [00:19<00:00, 336.62it/s]


In [4]:
plip_results['bondtype'].value_counts()

bondtype
Hydrogen Bonds              21539
pi-Stacking                  7354
Hydrophobic Interactions       86
Halogen Bonds                   1
Name: count, dtype: int64

In [8]:
plip_results

,resnr,restype,reschain,resnr_lig,restype_lig,reschain_lig,bondtype,item_id
0,32,G,A,102,ACT,A,Hydrogen Bonds,4fen_1_RNA_A.pdb_4fen_1_ACT_C.pdb
1,38,G,A,102,ACT,A,Hydrogen Bonds,4fen_1_RNA_A.pdb_4fen_1_ACT_C.pdb
0,47,U,A,101,HPA,A,Hydrogen Bonds,4fen_1_RNA_A.pdb_4fen_1_HPA_B.pdb
1,50,C,A,101,HPA,A,Hydrogen Bonds,4fen_1_RNA_A.pdb_4fen_1_HPA_B.pdb
2,51,U,A,101,HPA,A,Hydrogen Bonds,4fen_1_RNA_A.pdb_4fen_1_HPA_B.pdb
...,...,...,...,...,...,...,...,...
7,29,A,A,101,PRF,A,Hydrogen Bonds,6vui_1_RNA_A.pdb_6vui_1_PRF_B.pdb
8,5,G,A,101,PRF,A,pi-Stacking,6vui_1_RNA_A.pdb_6vui_1_PRF_B.pdb
9,11,G,A,101,PRF,A,pi-Stacking,6vui_1_RNA_A.pdb_6vui_1_PRF_B.pdb
10,11,G,A,101,PRF,A,pi-Stacking,6vui_1_RNA_A.pdb_6vui_1_PRF_B.pdb


In [11]:
test_data_ids = [item['id'] for item in test_data]
test_data[test_data_ids.index('4fen_1_RNA_A.pdb_4fen_1_ACT_C.pdb')]['block_to_pdb_indexes']

{1: 'A_38', 2: 'A_39', 3: 'A_66', 4: 'A_67'}